<a href="https://colab.research.google.com/github/d3m1d0s/FoML_projects/blob/main/project_1/bankchurners_eda_clustering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Fundamentals of Machine Learning, Project 1**

### Exploratory and Cluster Analysis: Credit Card Customers

Author: Demid Ostiakov (OST0059)  
Dataset: [Credit Card Customers (Kaggle)](https://www.kaggle.com/datasets/sakshigoyal7/credit-card-customers)

### Import libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

### Import data

In [2]:
DATA_URL = "https://raw.githubusercontent.com/d3m1d0s/FoML_projects/main/project_1/BankChurners.csv/BankChurners.csv"

df = pd.read_csv(DATA_URL)

<div style="background-color: #05c3de; padding: 20px;">
    <h2><font color='white'> First Look at the Data </font></h2>
</div>

Display the first few rows of the dataset.

In [3]:
df.head()

,CLIENTNUM,Attrition_Flag,Customer_Age,Gender,Dependent_count,Education_Level,Marital_Status,Income_Category,Card_Category,Months_on_book,...,Credit_Limit,Total_Revolving_Bal,Avg_Open_To_Buy,Total_Amt_Chng_Q4_Q1,Total_Trans_Amt,Total_Trans_Ct,Total_Ct_Chng_Q4_Q1,Avg_Utilization_Ratio,Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_1,Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_2
0,768805383,Existing Customer,45,M,3,High School,Married,$60K - $80K,Blue,39,...,12691.0,777,11914.0,1.335,1144,42,1.625,0.061,0.000093,0.99991
1,818770008,Existing Customer,49,F,5,Graduate,Single,Less than $40K,Blue,44,...,8256.0,864,7392.0,1.541,1291,33,3.714,0.105,0.000057,0.99994
2,713982108,Existing Customer,51,M,3,Graduate,Married,$80K - $120K,Blue,36,...,3418.0,0,3418.0,2.594,1887,20,2.333,0.000,0.000021,0.99998
3,769911858,Existing Customer,40,F,4,High School,Unknown,Less than $40K,Blue,34,...,3313.0,2517,796.0,1.405,1171,20,2.333,0.760,0.000134,0.99987
4,709106358,Existing Customer,40,M,3,Uneducated,Married,$60K - $80K,Blue,21,...,4716.0,0,4716.0,2.175,816,28,2.500,0.000,0.000022,0.99998


Each row describes one credit card customer of the bank: who they are (age, gender, education, income), which card they hold, and how they used it over the last 12 months.

The target is `Attrition_Flag`. "Attrited Customer" means the customer closed the account, while "Existing Customer" means they stayed. In this part of the project it will not be used to build clusters, only to describe them afterwards.

Check the basic information about the DataFrame.

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10127 entries, 0 to 10126
Data columns (total 23 columns):
 #   Column                                                                                                                              Non-Null Count  Dtype  
---  ------                                                                                                                              --------------  -----  
 0   CLIENTNUM                                                                                                                           10127 non-null  int64  
 1   Attrition_Flag                                                                                                                      10127 non-null  object 
 2   Customer_Age                                                                                                                        10127 non-null  int64  
 3   Gender                                                                           

The dataset has 10,127 rows and 23 columns. According to `info()`, there are no missing values, as every column has 10,127 non-null entries. However, `head()` already shows the value "Unknown" in `Marital_Status` (row 3), so some missing information may be stored as text instead of NaN. This will be checked separately.

Some numerical columns are counts rather than continuous measurements:
- `Dependent_count`, `Total_Relationship_Count`, `Months_Inactive_12_mon` and `Contacts_Count_12_mon` take only a few values (0 to 6).
- `Total_Trans_Ct` and `Months_on_book` are counts too, but their range is wide enough that they behave almost like continuous variables.
- `CLIENTNUM` is stored as `int64`, but it is only an identifier and its value carries no numerical meaning.

Three columns are not real customer features and will be removed:
- `CLIENTNUM`, because it only identifies the customer.
- The two `Naive_Bayes_Classifier_...` columns. They are the output of a classifier trained on `Attrition_Flag`, so they effectively contain the answer. The dataset author also recommends deleting them.